# Physics-Driven Motion on Pretrained Gaussian Splats

This notebook contains the original four experiments for *Dream Worlds: Physics Simulation on Gaussian Splats*, developed for Georgia Tech's Spring 2026 CGAI course. It applies simple physics-driven motion rules to a pretrained Gaussian Splatting representation of the Ficus scene.

The goal is not to build a fully physically accurate simulator, but to study how direct position updates affect the rendered appearance of a recognizable Gaussian Splat object across controlled motion baselines. See the [repository README](../README.md), [qualitative results](../docs/RESULTS.md), and [technical report](../paper/dream-worlds-technical-report.pdf) for the complete project context.

## Execution contract

This notebook requires Linux or WSL, an NVIDIA CUDA environment, FFmpeg, and the pretrained Ficus scene under `output/ficus_whitebg-trained/`. The required inputs are `cameras.json` and `point_cloud/iteration_30000/point_cloud.ply`. Each experiment reloads the original checkpoint before applying its own update rule.

Run the cells from top to bottom with the `gaussian_splatting` kernel. Rendered PNG frames are written under `assets/images/<scene_name>/images/<experiment_name>/`, and the export cells convert those sequences into MP4 videos under `assets/demos/`. Existing videos with the same filenames are overwritten.

In [ ]:
import os
from pathlib import Path

import torchvision
from gaussian_splatting import GaussianModel, Camera
import torch


CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT = CURRENT_DIR
if not (REPO_ROOT / "output").exists() and (CURRENT_DIR.parent / "output").exists():
    REPO_ROOT = CURRENT_DIR.parent
os.chdir(REPO_ROOT)

SCENE_ROOT = REPO_ROOT / "output" / "ficus_whitebg-trained"
SCENE_NAME = SCENE_ROOT.name
FRAMES_ROOT = REPO_ROOT / "assets" / "images" / SCENE_NAME / "images"
CAMERAS_PATH = SCENE_ROOT / "cameras.json"
POINT_CLOUD_PATH = SCENE_ROOT / "point_cloud" / "iteration_30000" / "point_cloud.ply"
DEMO_DIR = REPO_ROOT / "assets" / "demos"

def render(gaussians: GaussianModel, camera: Camera, camera_id: int, pyhs_iter: int, save: str):
    # Render the current Gaussian state and save the frame to disk.
    os.makedirs(save, exist_ok=True)
    out = gaussians(camera)
    rendering = out['render']
    torchvision.utils.save_image(rendering, os.path.join(save, f'{camera_id}' + '_{0:05d}'.format(pyhs_iter) + ".png"))

## Experiment 1: Uniform Gravity With Position Clamp

This baseline applies the same gravity vector to every Gaussian and clamps the updated positions to a bounded spatial region.

It serves as a reference case for globally consistent motion and helps show how a simple shared update rule affects the rendered structure of the object.

In [ ]:
from gaussian_splatting.dataset import JSONCameraDataset

# Experiment configuration
dt = .02
camera_id = 0
model_results = str(FRAMES_ROOT / "wall_smash")

# Load the pretrained Gaussian representation and define the gravity direction
gaussians = GaussianModel(3).to('cuda')
gravity = torch.tensor([0., -1., 0.], device=gaussians.get_xyz.device)

# Load camera metadata for the pretrained ficus scene
cameras = JSONCameraDataset(str(CAMERAS_PATH)).to('cuda')

# Load the pretrained checkpoint used for this experiment
gaussians.load_ply(str(POINT_CLOUD_PATH))

# Allocate per-Gaussian state and run the simulation loop
n = gaussians.get_xyz.shape[0]
camera = cameras[camera_id]
velocity = torch.zeros_like(gaussians.get_xyz, device=gaussians.get_xyz.device)

for step in range(200):
    with torch.no_grad():
        # Apply a constant gravity update to the velocity
        velocity += gravity * dt
        # Keep the motion inside a bounded region
        points = torch.clamp(gaussians.get_xyz + velocity * dt, min=-5., max=5.)
        gaussians.get_xyz.data.copy_(points)

        # Render and save the current simulation frame
        frame = render(gaussians, camera, camera_id, step, model_results)

### Export Experiment 1 Video

The following cell converts the rendered PNG sequence from Experiment 1 into an MP4 file for qualitative inspection.

In [ ]:
# Export the rendered frame sequence for Experiment 1
!ffmpeg -i {model_results}/{camera_id}_%05d.png -vcodec libx264 -preset slow -qp 0 -r 60 -threads 0 {str(DEMO_DIR / 'wall_smash.mp4')} -y

## Experiment 2: Randomized Inverse-Mass Motion

This baseline keeps the same rendering pipeline but scales motion by a randomized inverse-mass term for each Gaussian.

Unlike Experiment 1, this produces nonuniform motion across the object and makes local variation more visible in the rendered result.

In [ ]:
# Experiment configuration
dt = .02
camera_id = 2
model_results = str(FRAMES_ROOT / "mass_falling")

# Load the pretrained Gaussian representation and define the gravity direction
gaussians = GaussianModel(3).to('cuda')
gravity = torch.tensor([0., 0., -1.], device=gaussians.get_xyz.device)

# Load camera metadata for the pretrained ficus scene
cameras = JSONCameraDataset(str(CAMERAS_PATH)).to('cuda')

# Load the pretrained checkpoint used for this experiment
gaussians.load_ply(str(POINT_CLOUD_PATH))

# Allocate per-Gaussian state and run the simulation loop
n = gaussians.get_xyz.shape[0]

camera = cameras[camera_id]
inv_mass = 1 / torch.rand((n, 1), device=gaussians.get_xyz.device)

velocity = torch.zeros_like(gaussians.get_xyz, device=gaussians.get_xyz.device)

for step in range(200):
    with torch.no_grad():
        # Apply a constant gravity update to the velocity
        velocity += gravity * dt
        # Scale the position update by randomized inverse mass
        points = torch.clamp(gaussians.get_xyz + velocity * inv_mass * dt, min=-10., max=10.)
        gaussians.get_xyz.data.copy_(points)

        # Render and save the current simulation frame
        frame = render(gaussians, camera, camera_id, step, model_results)

### Export Experiment 2 Video

The following cell converts the rendered PNG sequence from Experiment 2 into an MP4 file for qualitative comparison against the uniform-gravity baseline.

In [ ]:
# Export the rendered frame sequence for Experiment 2
!ffmpeg -i {model_results}/{camera_id}_%05d.png -vcodec libx264 -preset slow -qp 0 -r 60 -threads 0 {str(DEMO_DIR / 'mass_falling.mp4')} -y

## Experiment 3: Wind Field Deformation

This baseline applies a smooth wind field to the Gaussian cloud and uses light velocity damping to avoid unbounded drift.

It serves as a coherent external-force case and helps show how lateral flow changes the rendered structure of the object.

In [ ]:
# Experiment configuration
dt = .02
camera_id = 0
model_results = str(FRAMES_ROOT / "wind_field")

# Load the pretrained Gaussian representation and define the wind setup
gaussians = GaussianModel(3).to('cuda')
gravity = torch.tensor([0., -0.15, 0.], device=gaussians.get_xyz.device)

# Load camera metadata for the pretrained ficus scene
cameras = JSONCameraDataset(str(CAMERAS_PATH)).to('cuda')

# Load the pretrained checkpoint used for this experiment
gaussians.load_ply(str(POINT_CLOUD_PATH))

# Allocate per-Gaussian state and run the simulation loop
n = gaussians.get_xyz.shape[0]
camera = cameras[camera_id]
velocity = torch.zeros_like(gaussians.get_xyz, device=gaussians.get_xyz.device)

for step in range(200):
    with torch.no_grad():
        positions = gaussians.get_xyz
        # Apply a smooth wind field with light damping to keep the motion stable
        gust_x = 0.8 + 0.3 * torch.sin(2.0 * positions[:, 1:2] + step * 0.08)
        gust_z = 0.15 * torch.cos(1.5 * positions[:, 0:1] + step * 0.05)
        wind = torch.cat((gust_x, torch.zeros_like(gust_x), gust_z), dim=1)
        velocity = 0.98 * velocity + (gravity + wind) * dt
        points = torch.clamp(positions + velocity * dt, min=-8., max=8.)
        gaussians.get_xyz.data.copy_(points)

        # Render and save the current simulation frame
        frame = render(gaussians, camera, camera_id, step, model_results)

### Export Experiment 3 Video

The following cell converts the rendered PNG sequence from Experiment 3 into an MP4 file for qualitative inspection of the wind-driven deformation.

In [ ]:
# Export the rendered frame sequence for Experiment 3
!ffmpeg -i {model_results}/{camera_id}_%05d.png -vcodec libx264 -preset slow -qp 0 -r 60 -threads 0 {str(DEMO_DIR / 'wind_field.mp4')} -y

## Experiment 4: Wind Strength Ablation

This experiment reuses the wind-field setup from Experiment 3 and varies only the wind magnitude.

We treat Experiment 3 as the medium-wind baseline and compare it against lower and higher wind strengths to study how force magnitude affects visual coherence. In this notebook, the low-wind variant uses `wind_scale = 0.1`, the medium-wind baseline corresponds to Experiment 3, and the high-wind variant uses `wind_scale = 4.0`.

### Low Wind Variant

This variant reduces the wind magnitude while keeping the rest of the pipeline unchanged.

In [ ]:
# Experiment configuration
dt = .02
camera_id = 0
wind_scale = 0.1
model_results = str(FRAMES_ROOT / "wind_field_low")

# Load the pretrained Gaussian representation and define the wind setup
gaussians = GaussianModel(3).to('cuda')
gravity = torch.tensor([0., -0.15, 0.], device=gaussians.get_xyz.device)

# Load camera metadata for the pretrained ficus scene
cameras = JSONCameraDataset(str(CAMERAS_PATH)).to('cuda')

# Load the pretrained checkpoint used for this experiment
gaussians.load_ply(str(POINT_CLOUD_PATH))

# Allocate per-Gaussian state and run the simulation loop
n = gaussians.get_xyz.shape[0]
camera = cameras[camera_id]
velocity = torch.zeros_like(gaussians.get_xyz, device=gaussians.get_xyz.device)

for step in range(200):
    with torch.no_grad():
        positions = gaussians.get_xyz
        # Apply the Experiment 3 wind field at reduced strength
        gust_x = wind_scale * (0.8 + 0.3 * torch.sin(2.0 * positions[:, 1:2] + step * 0.08))
        gust_z = wind_scale * (0.15 * torch.cos(1.5 * positions[:, 0:1] + step * 0.05))
        wind = torch.cat((gust_x, torch.zeros_like(gust_x), gust_z), dim=1)
        velocity = 0.98 * velocity + (gravity + wind) * dt
        points = torch.clamp(positions + velocity * dt, min=-8., max=8.)
        gaussians.get_xyz.data.copy_(points)

        # Render and save the current simulation frame
        frame = render(gaussians, camera, camera_id, step, model_results)

### Export Low Wind Video

The following cell converts the rendered PNG sequence from the low-wind variant into an MP4 file.

In [ ]:
# Export the rendered frame sequence for the low-wind variant
!ffmpeg -i {model_results}/{camera_id}_%05d.png -vcodec libx264 -preset slow -qp 0 -r 60 -threads 0 {str(DEMO_DIR / 'wind_field_low.mp4')} -y

### High Wind Variant

This variant increases the wind magnitude while keeping the rest of the pipeline unchanged.

In [ ]:
# Experiment configuration
dt = .02
camera_id = 0
wind_scale = 4.0
model_results = str(FRAMES_ROOT / "wind_field_high")

# Load the pretrained Gaussian representation and define the wind setup
gaussians = GaussianModel(3).to('cuda')
gravity = torch.tensor([0., -0.15, 0.], device=gaussians.get_xyz.device)

# Load camera metadata for the pretrained ficus scene
cameras = JSONCameraDataset(str(CAMERAS_PATH)).to('cuda')

# Load the pretrained checkpoint used for this experiment
gaussians.load_ply(str(POINT_CLOUD_PATH))

# Allocate per-Gaussian state and run the simulation loop
n = gaussians.get_xyz.shape[0]
camera = cameras[camera_id]
velocity = torch.zeros_like(gaussians.get_xyz, device=gaussians.get_xyz.device)

for step in range(200):
    with torch.no_grad():
        positions = gaussians.get_xyz
        # Apply the Experiment 3 wind field at increased strength
        gust_x = wind_scale * (0.8 + 0.3 * torch.sin(2.0 * positions[:, 1:2] + step * 0.08))
        gust_z = wind_scale * (0.15 * torch.cos(1.5 * positions[:, 0:1] + step * 0.05))
        wind = torch.cat((gust_x, torch.zeros_like(gust_x), gust_z), dim=1)
        velocity = 0.98 * velocity + (gravity + wind) * dt
        points = torch.clamp(positions + velocity * dt, min=-8., max=8.)
        gaussians.get_xyz.data.copy_(points)

        # Render and save the current simulation frame
        frame = render(gaussians, camera, camera_id, step, model_results)

### Export High Wind Video

The following cell converts the rendered PNG sequence from the high-wind variant into an MP4 file.

In [ ]:
# Export the rendered frame sequence for the high-wind variant
!ffmpeg -i {model_results}/{camera_id}_%05d.png -vcodec libx264 -preset slow -qp 0 -r 60 -threads 0 {str(DEMO_DIR / 'wind_field_high.mp4')} -y